# Schema Validation Exercise 3: analysis and adjudication

This notebook mirrors the Validation 2 analysis workflow for the completed Schema v1.2 run. It checks run integrity, summarizes configuration and spend, exposes model-screened candidates, and prepares an empty worksheet for bounded human adjudication.

Boundaries:

- A surfaced candidate is not an accepted schema gap.
- Only surfaced candidates are reviewed; no-gap results are not audited and recall is not estimated.
- Calibration outputs are separate from substantive counts.
- The Validation 2 comparison is descriptive and cannot isolate a causal effect of Markdown versus Pydantic.
- Pricing uses current GPT-5.5 Batch rates because Flex is billed at Batch API rates. Recheck official pricing before using the estimate as an accounting record.


In [ ]:
from __future__ import annotations

import json
from collections.abc import Iterable
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Image, display


## Configuration

All paths are repository-relative. This notebook reads completed JSONL outputs and images; it makes no API calls and writes no files.


In [ ]:
from schema_development.paths import ROOT as REPO_ROOT

OUTPUTS_DIR = REPO_ROOT / "artifacts/appendix_h"
RESULTS_PATH = OUTPUTS_DIR / "results.jsonl"
ERRORS_PATH = OUTPUTS_DIR / "errors.jsonl"
CALIBRATION0_PATH = OUTPUTS_DIR / "calibration_results0.jsonl"
CALIBRATION0_ERRORS_PATH = OUTPUTS_DIR / "calibration_errors0.jsonl"
CALIBRATION1_PATH = OUTPUTS_DIR / "calibration_results1.jsonl"
CALIBRATION1_ERRORS_PATH = OUTPUTS_DIR / "calibration_errors1.jsonl"
SNAPSHOTS_DIR = REPO_ROOT / "data/source/heldout/snapshots"
VALIDATION2_RESULTS_PATH = REPO_ROOT / "artifacts/validation2/results.jsonl"
PRICES_USD_PER_1M = {
    ("gpt-5.5", "flex"): {
        "input": 5.00,
        "cached_input": 0.50,
        "output": 30.00,
    }
}


In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    """Load JSONL rows or return an empty list for an absent file.

    Parameters
    ----------
    path : Path
        JSONL file to read.

    Returns
    -------
    list[dict[str, Any]]
        Parsed JSON objects in file order.
    """
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def get_service_tier(record: dict[str, Any]) -> str:
    """Return a result record's configured service tier.

    Parameters
    ----------
    record : dict[str, Any]
        Validation result record.

    Returns
    -------
    str
        Lower-case service-tier name.
    """
    return str(record.get("request_config", {}).get("service_tier", "")).lower()


def calculate_cost_usd(record: dict[str, Any]) -> dict[str, float | int]:
    """Calculate token quantities and estimated API cost for one record.

    Parameters
    ----------
    record : dict[str, Any]
        Result record containing model, tier, and usage.

    Returns
    -------
    dict[str, float | int]
        Token quantities and estimated cost.

    Raises
    ------
    KeyError
        If pricing is unavailable for the model and tier.
    """
    key = (str(record.get("model", "")), get_service_tier(record))
    prices = PRICES_USD_PER_1M[key]
    usage = record.get("usage", {})
    input_tokens = int(usage.get("input_tokens", 0) or 0)
    cached_tokens = int(
        usage.get("input_tokens_details", {}).get("cached_tokens", 0) or 0
    )
    output_tokens = int(usage.get("output_tokens", 0) or 0)
    uncached_tokens = max(input_tokens - cached_tokens, 0)
    cost_usd = (
        uncached_tokens * prices["input"]
        + cached_tokens * prices["cached_input"]
        + output_tokens * prices["output"]
    ) / 1_000_000
    return {
        "input_tokens": input_tokens,
        "cached_tokens": cached_tokens,
        "uncached_tokens": uncached_tokens,
        "output_tokens": output_tokens,
        "cost_usd": cost_usd,
    }


def expected_assessment(gaps: list[dict[str, Any]]) -> str:
    """Infer the required assessment from candidate statuses.

    Parameters
    ----------
    gaps : list[dict[str, Any]]
        Candidate gaps for one snapshot.

    Returns
    -------
    str
        Assessment implied by candidate statuses.
    """
    if not gaps:
        return "no_critical_gap_found"
    if any(gap.get("gap_status") == "critical" for gap in gaps):
        return "critical_gap_found"
    return "possible_gap"


## Load and normalize results


In [ ]:
results = load_jsonl(RESULTS_PATH)
errors = load_jsonl(ERRORS_PATH)
calibration0_results = load_jsonl(CALIBRATION0_PATH)
calibration0_errors = load_jsonl(CALIBRATION0_ERRORS_PATH)
calibration1_results = load_jsonl(CALIBRATION1_PATH)
calibration1_errors = load_jsonl(CALIBRATION1_ERRORS_PATH)
validation2_results = load_jsonl(VALIDATION2_RESULTS_PATH)

result_rows: list[dict[str, Any]] = []
gap_rows: list[dict[str, Any]] = []
for record in results:
    parsed = record.get("parsed_output", {})
    gaps = parsed.get("critical_or_possible_gaps", [])
    result_rows.append(
        {
            "snapshot_file_name": record.get("snapshot_file_name"),
            "source": record.get("source"),
            "artifact_type": record.get("artifact_type"),
            "schema_version": record.get("schema_version"),
            "coverage_assessment": parsed.get("coverage_assessment"),
            "candidate_count": len(gaps),
            "elapsed_seconds": record.get("elapsed_seconds", 0.0),
        }
    )
    for gap in gaps:
        gap_rows.append(
            {
                "snapshot_file_name": record.get("snapshot_file_name"),
                "source": record.get("source"),
                "artifact_type": record.get("artifact_type"),
                "source_document_id": record.get("source_document_id"),
                "snapshot_path": record.get("snapshot_path"),
                **gap,
            }
        )
results_df = pd.DataFrame(result_rows)
gaps_df = pd.DataFrame(gap_rows)
print(f"Loaded {len(results):,} substantive results and {len(errors):,} errors.")
print(
    f"Surfaced {len(gaps_df):,} candidates across {gaps_df['snapshot_file_name'].nunique():,} snapshots."
)


## Run-integrity gate

These checks must pass before outputs are interpreted. Expected inputs are the same 202 PNGs used by Validation 2.


In [ ]:
expected_names = sorted(path.name for path in SNAPSHOTS_DIR.rglob("*.png"))
result_names = [record["snapshot_file_name"] for record in results]
validation2_names = [record["snapshot_file_name"] for record in validation2_results]
integrity_checks = {
    "202 substantive inputs discovered": len(expected_names) == 202,
    "202 substantive results present": len(results) == 202,
    "No substantive error rows": not errors,
    "Result filenames are unique": len(result_names) == len(set(result_names)),
    "Results exactly match source PNGs": set(result_names) == set(expected_names),
    "Results exactly match Validation 2": set(result_names) == set(validation2_names),
    "All API responses completed": all(
        record.get("api_status") == "completed" for record in results
    ),
    "All results evaluate Schema v1.2": all(
        record.get("schema_version") == "1.2" for record in results
    ),
    "No schema fields were excluded": all(
        record.get("excluded_schema_fields") == [] for record in results
    ),
    "Assessments agree with candidate statuses": all(
        record.get("parsed_output", {}).get("coverage_assessment")
        == expected_assessment(
            record.get("parsed_output", {}).get("critical_or_possible_gaps", [])
        )
        for record in results
    ),
}
integrity_df = pd.DataFrame(
    [
        {"check": key, "status": "PASS" if value else "FAIL"}
        for key, value in integrity_checks.items()
    ]
)
display(integrity_df)
assert all(integrity_checks.values()), "Resolve failed integrity checks first."


In [ ]:
configuration_df = pd.DataFrame(
    [
        {
            "declared_model": record.get("model"),
            "resolved_model": record.get("raw_response", {}).get("model"),
            "reasoning_effort": record.get("request_config", {})
            .get("reasoning", {})
            .get("effort"),
            "service_tier": get_service_tier(record),
            "max_output_tokens": record.get("request_config", {}).get(
                "max_output_tokens"
            ),
            "prompt_cache_key": record.get("request_config", {}).get(
                "prompt_cache_key"
            ),
        }
        for record in results
    ]
).drop_duplicates()
display(configuration_df)


## API usage, runtime, and estimated spend

The estimate applies current GPT-5.5 Batch prices to Flex usage: $5.00 per million uncached input tokens, $0.50 per million cached input tokens, and $30.00 per million output tokens. Output includes reasoning tokens. Pricing may change.


In [ ]:
cost_df = pd.DataFrame(
    [
        {
            "snapshot_file_name": record["snapshot_file_name"],
            "source": record.get("source"),
            "artifact_type": record.get("artifact_type"),
            "elapsed_seconds": float(record.get("elapsed_seconds", 0.0) or 0.0),
            **calculate_cost_usd(record),
        }
        for record in results
    ]
)
usage_summary = pd.DataFrame(
    [
        {
            "requests": len(cost_df),
            "input_tokens": int(cost_df["input_tokens"].sum()),
            "cached_tokens": int(cost_df["cached_tokens"].sum()),
            "uncached_tokens": int(cost_df["uncached_tokens"].sum()),
            "output_tokens": int(cost_df["output_tokens"].sum()),
            "summed_elapsed_minutes": cost_df["elapsed_seconds"].sum() / 60,
            "estimated_cost_usd": cost_df["cost_usd"].sum(),
        }
    ]
)
display(usage_summary.style.format({"estimated_cost_usd": "USD {:,.6f}"}))


In [ ]:
cost_by_source = (
    cost_df.groupby("source", dropna=False)
    .agg(
        requests=("snapshot_file_name", "count"),
        input_tokens=("input_tokens", "sum"),
        cached_tokens=("cached_tokens", "sum"),
        output_tokens=("output_tokens", "sum"),
        estimated_cost_usd=("cost_usd", "sum"),
    )
    .reset_index()
)
display(cost_by_source.style.format({"estimated_cost_usd": "USD {:,.6f}"}))


## Full-run screening summary

These are pre-adjudication model-screening outputs. Candidate counts are not confirmed missing fields.


In [ ]:
assessment_order = ["no_critical_gap_found", "possible_gap", "critical_gap_found"]
assessment_summary = (
    results_df["coverage_assessment"]
    .value_counts()
    .reindex(assessment_order, fill_value=0)
    .rename_axis("coverage_assessment")
    .reset_index(name="snapshots")
)
assessment_summary["share"] = assessment_summary["snapshots"] / len(results_df)
display(assessment_summary.style.format({"share": "{:.1%}"}))
display(
    pd.DataFrame(
        [
            {
                "candidate_snapshots": gaps_df["snapshot_file_name"].nunique(),
                "candidate_rows": len(gaps_df),
                "possible_candidates": int((gaps_df["gap_status"] == "possible").sum()),
                "critical_candidates": int((gaps_df["gap_status"] == "critical").sum()),
            }
        ]
    )
)


In [ ]:
display(
    pd.crosstab(
        results_df["source"], results_df["coverage_assessment"], margins=True
    ).reindex(columns=assessment_order + ["All"], fill_value=0)
)
display(
    pd.crosstab(
        results_df["artifact_type"], results_df["coverage_assessment"], margins=True
    ).reindex(columns=assessment_order + ["All"], fill_value=0)
)


## Candidate review tables

These tables organize reported candidates without accepting model classifications or proposed field names.


In [ ]:
candidate_columns = [
    "snapshot_file_name",
    "source",
    "artifact_type",
    "gap_id",
    "gap_status",
    "missing_metadata_concept",
    "proposed_field_name",
    "evidence",
    "why_snapshot_metadata",
    "closest_schema_paths",
    "why_existing_fields_may_be_insufficient",
    "material_impact",
    "material_consequence",
    "uncertainty_note",
]
display(gaps_df[candidate_columns])


In [ ]:
for title, table in {
    "Gap status": gaps_df["gap_status"].value_counts(),
    "Material impact": gaps_df["material_impact"].value_counts(),
    "Proposed field name": gaps_df["proposed_field_name"].value_counts(),
}.items():
    print(title)
    display(table.rename("candidates").to_frame())

closest_path_summary = (
    gaps_df[["closest_schema_paths"]]
    .explode("closest_schema_paths")
    .dropna()
    .value_counts()
    .rename("candidates")
    .reset_index()
)
display(closest_path_summary)


## Descriptive comparison with Validation 2

This aligns snapshot-level assessments across the runs. It is diagnostic only: differences may reflect the schema, representation, or ordinary model-run variability, and this design does not identify their separate effects.


In [ ]:
validation2_assessments = {
    record["snapshot_file_name"]: record.get("parsed_output", {}).get(
        "coverage_assessment"
    )
    for record in validation2_results
}
comparison_df = results_df[["snapshot_file_name", "coverage_assessment"]].copy()
comparison_df["validation2_assessment"] = comparison_df["snapshot_file_name"].map(
    validation2_assessments
)
comparison_df = comparison_df.rename(
    columns={"coverage_assessment": "validation3_assessment"}
)
assessment_comparison = pd.crosstab(
    comparison_df["validation2_assessment"], comparison_df["validation3_assessment"]
).reindex(index=assessment_order, columns=assessment_order, fill_value=0)
display(assessment_comparison)


## Human adjudication worksheet

Review every surfaced candidate against the protocol. Suggested decision labels are:

- covered_by_v1_2
- outside_snapshot_metadata_boundary
- extraction_content
- insufficient_evidence_or_materiality
- documentation_or_representation_concern
- meaningful_missing_field

Enter decisions in HUMAN_REVIEW, keyed by (snapshot_file_name, gap_id). The mapping is intentionally empty. Re-running the cell merges entries into the worksheet without changing raw outputs.


In [ ]:
HUMAN_REVIEW: dict[tuple[str, str], dict[str, Any]] = {
    # ("example_snapshot.png", "gap_1"): {
    #     "human_decision": "covered_by_v1_2",
    #     "critical_to_interpretability": False,
    #     "critical_to_discoverability": False,
    #     "human_rationale": "Explain the adjudication.",
    # },
}

review_df = gaps_df[candidate_columns].copy()
review_df["human_decision"] = ""
review_df["critical_to_interpretability"] = pd.NA
review_df["critical_to_discoverability"] = pd.NA
review_df["human_rationale"] = ""
for index, row in review_df.iterrows():
    key = (row["snapshot_file_name"], row["gap_id"])
    for field, value in HUMAN_REVIEW.get(key, {}).items():
        if field in review_df.columns:
            review_df.at[index, field] = value
display(review_df.sort_values("snapshot_file_name"))


### Review sequence

1. Inspect the snapshot and evidence.
2. Decide whether the concept is reusable snapshot metadata rather than document context or extraction content.
3. Test exact closest_schema_paths against v1.2 semantics.
4. Decide whether any loss is material to interpretation, discoverability, or both.
5. Record one decision and concise rationale.

Do not infer review outcomes from gap_status or a proposed field name.


## Calibration controls

Calibration0 tested the complete v1.2 schema on 12 snapshots. Calibration1 was the provenance sensitivity control with provenance and interpretive_notes removed from a local schema copy. Calibration is excluded from substantive summaries.


In [ ]:
def summarize_run(
    records: Iterable[dict[str, Any]], run_name: str, error_count: int
) -> dict[str, Any]:
    """Summarize one calibration result collection.

    Parameters
    ----------
    records : Iterable[dict[str, Any]]
        Result records to summarize.
    run_name : str
        Human-readable run label.
    error_count : int
        Number of associated error rows.

    Returns
    -------
    dict[str, Any]
        Run-level result and candidate counts.
    """
    rows = list(records)
    assessments = [
        row.get("parsed_output", {}).get("coverage_assessment") for row in rows
    ]
    gaps = [
        gap
        for row in rows
        for gap in row.get("parsed_output", {}).get("critical_or_possible_gaps", [])
    ]
    return {
        "run": run_name,
        "results": len(rows),
        "errors": error_count,
        "no_critical_gap_found": assessments.count("no_critical_gap_found"),
        "possible_gap": assessments.count("possible_gap"),
        "critical_gap_found": assessments.count("critical_gap_found"),
        "candidate_rows": len(gaps),
    }


calibration_summary = pd.DataFrame(
    [
        summarize_run(
            calibration0_results,
            "Calibration0: complete v1.2 schema",
            len(calibration0_errors),
        ),
        summarize_run(
            calibration1_results,
            "Calibration1: provenance ablation",
            len(calibration1_errors),
        ),
    ]
)
display(calibration_summary)

calibration_candidate_rows = []
for run_name, records in (
    ("Calibration0: complete v1.2 schema", calibration0_results),
    ("Calibration1: provenance ablation", calibration1_results),
):
    for record in records:
        for gap in record.get("parsed_output", {}).get("critical_or_possible_gaps", []):
            calibration_candidate_rows.append(
                {
                    "run": run_name,
                    "snapshot_file_name": record.get("snapshot_file_name"),
                    **gap,
                }
            )
display(pd.DataFrame(calibration_candidate_rows))


## Inspect candidate snapshots

Use inspect_gap to display a snapshot and all its candidate rows. It uses the recorded path first and falls back to the repository snapshot tree.


In [ ]:
def resolve_snapshot_path(snapshot_file_name: str) -> Path:
    """Resolve a candidate snapshot to an existing image path.

    Parameters
    ----------
    snapshot_file_name : str
        Snapshot filename stored in a result.

    Returns
    -------
    Path
        Existing path to the image.

    Raises
    ------
    FileNotFoundError
        If the snapshot cannot be resolved uniquely.
    """
    rows = gaps_df[gaps_df["snapshot_file_name"] == snapshot_file_name]
    if not rows.empty:
        recorded = Path(str(rows.iloc[0]["snapshot_path"]))
        if recorded.exists():
            return recorded
    matches = list(SNAPSHOTS_DIR.rglob(snapshot_file_name))
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        f"Expected one image for {snapshot_file_name!r}; found {len(matches)}."
    )


def inspect_gap(snapshot_file_name: str, width: int = 1100) -> None:
    """Display one snapshot and its candidate gaps.

    Parameters
    ----------
    snapshot_file_name : str
        Snapshot filename to inspect.
    width : int, default=1100
        Display width in pixels.

    Returns
    -------
    None
        Displays the image and candidate table.
    """
    rows = gaps_df[gaps_df["snapshot_file_name"] == snapshot_file_name]
    if rows.empty:
        raise KeyError(f"No candidate rows found for {snapshot_file_name!r}.")
    display(Image(filename=str(resolve_snapshot_path(snapshot_file_name)), width=width))
    display(rows[candidate_columns])


In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
gap_snapshots = (
    gaps_df.groupby(["snapshot_file_name", "source", "artifact_type"], dropna=False)
    .agg(
        candidate_count=("gap_id", "count"),
        statuses=("gap_status", lambda values: ", ".join(sorted(set(values)))),
        proposed_fields=(
            "proposed_field_name",
            lambda values: ", ".join(sorted(set(values))),
        ),
    )
    .reset_index()
)
display(gap_snapshots)


In [ ]:
selected_snapshot = gap_snapshots.iloc[20]["snapshot_file_name"]
inspect_gap(selected_snapshot)


## Full results overview

This compact table supports spot-checking and navigation across all 202 substantive results.


In [ ]:
overview_df = results_df.sort_values(
    ["coverage_assessment", "source", "artifact_type", "snapshot_file_name"]
).reset_index(drop=True)
display(overview_df)


## Unresolved errors

A successful completed run should leave this table empty.


In [ ]:
errors_df = pd.json_normalize(errors) if errors else pd.DataFrame()
display(errors_df)
